# Импорты

In [112]:
!pip install corus pymorphy3 nltk tqdm optuna gensim navec

In [2]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

--2026-04-21 15:49:36--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/0b363e00-0126-11e9-9e3c-e8c235463bd6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-21T16%3A40%3A22Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-21T15%3A39%3A53Z&ske=2026-04-21T16%3A40%3A22Z&sks=b&skv=2018-11-09&sig=DWLTpE3LWNDeQcb3fX0pmntUAIn%2BmUKdGJ5QwUDoO4w%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Njc5MDE3NywibmJmIjoxNzc2Nzg2NTc3LCJwYXRoIjoicmVsZWFzZWFzc2V0

In [133]:
import re
import numpy as np
import pandas as pd
import optuna

import nltk
import pymorphy3

from corus import load_lenta
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    classification_report,
)
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from tqdm import tqdm
from functools import lru_cache
from navec import Navec

# Загрузка данных

In [4]:
path = "lenta-ru-news.csv.gz"
records = load_lenta(path)

rows = []
for rec in records:
    rows.append({
        "title": rec.title,
        "text": rec.text,
        "topic": rec.topic,
    })

df = pd.DataFrame(rows)

# Оставим только подвыборку + выкинем непопулярные топики

df = df.sample(100000, replace=False).reset_index(drop=True)
topic_counts = df["topic"].value_counts().reset_index(drop=False)
need_topics = topic_counts[topic_counts['count']>1000].topic.unique()
df = df[df.topic.isin(need_topics)].reset_index(drop=True)

In [53]:
with pd.option_context("display.max_columns", None):
    display(df.sample(5))

,title,text,topic
72289,"Победитель ""Тур де Франс"" — 1997 дисквалифицир...",Спортивный арбитражный суд в Лозанне дисквалиф...,Спорт
119977,Буш рискованно пошутил о поисках оружия массов...,На прошедшем в среду ежегодном обеде с америка...,Мир
100094,Митволь потребует через суд остановки одного и...,Росприроднадзор намерен в судебном порядке пот...,Экономика
168449,Сенат США нашел деньги на производство истреби...,Комитет по делам вооруженных сил Сената США в ...,Наука и техника
56395,"""Аэрофлот"" собирается судиться с производителе...","Компания ""Аэрофлот – Российские авиалинии"" пыт...",Экономика


# Предобработка данных

Пайплайн преобработки: приведение к нижнему регистру, удаление возможных ссылок, email-ов, htlm-тегов, очистка от пунктуации и цифр, лемматизация (для снижения размера словаря), label-encoding на таргет (стандартный прием для многоклассовой классификации).

In [7]:
nltk.download("stopwords", quiet=True)
RUS_STOPWORDS = set(stopwords.words("russian"))
morph = pymorphy3.MorphAnalyzer()

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
USER_RE = re.compile(r"@\w+")
NUM_RE = re.compile(r"\b\d+([.,]\d+)?\b", re.UNICODE)
SPACE_RE = re.compile(r"\s+")
RU_WORD_RE = re.compile(r"[а-яё]+", re.IGNORECASE)


def normalize_text(text):
    text = text.lower().replace("ё", "е")
    text = URL_RE.sub(" ", text)
    text = USER_RE.sub(" ", text)
    text = NUM_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


@lru_cache(maxsize=200_000)
def lemmatize(token):
    if len(token) < 2:
        return ""
    if token in RUS_STOPWORDS:
        return ""
    parsed = morph.parse(token)[0]
    lemma = parsed.normal_form
    if lemma in RUS_STOPWORDS:
        return ""
    return lemma


def preprocess_text(text):
    text = normalize_text(text)
    tokens = RU_WORD_RE.findall(text)
    lemmas = [lemmatize(tok) for tok in tokens]
    lemmas = [t for t in lemmas if t]
    return " ".join(lemmas)


def preprocess_corpus(texts):
    return np.asarray([preprocess_text(t) for t in texts], dtype=object)


df["full_text"] = df["title"].astype(str) + " " + df["text"].astype(str)
df["processed_text"] = preprocess_corpus(df["full_text"])
df["topic"] = df["topic"].astype(str).str.strip()
label_encoder = LabelEncoder()
df["target"] = label_encoder.fit_transform(df["topic"])

In [9]:
df.head(3)

,title,text,topic,full_text,processed_text,target
0,Создан «умный» писсуар с экраном,Голландский стартап Mr.Friendly создал «умный»...,Наука и техника,Создан «умный» писсуар с экраном Голландский с...,создать умный писсуар экран голландский старта...,7
1,"Компания ""ЮКОС"" готова платить налоги по полно...","Глава нефтяной компании ""ЮКОС"" Семен Кукес зая...",Экономика,"Компания ""ЮКОС"" готова платить налоги по полно...",компания юкос готовый платить налог полный про...,12
2,В Северной Осетии перевернулся рейсовый автобус,В результате аварии рейсового автобуса в Север...,Россия,В Северной Осетии перевернулся рейсовый автобу...,северный осетия перевернуться рейсовый автобус...,8


# Train/eval/test-разбиение

In [13]:
X = df["processed_text"]
y = df["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Eval :", X_eval.shape, y_eval.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (59280,) (59280,)
Eval : (19760,) (19760,)
Test : (19761,) (19761,)


# Утилиты

In [134]:
def evaluate_model(model, X_part, y_part, labels=None):
    '''Делает прогноз обученной моделью и считает метрики классификации.'''
    y_pred = model.predict(X_part)

    metrics = {
        "accuracy": accuracy_score(y_part, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_part, y_pred),
        "precision_macro": precision_score(y_part, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_part, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_part, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_part, y_pred, average="weighted", zero_division=0),
    }

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_part)
        try:
            metrics["log_loss"] = log_loss(y_part, y_proba, labels=labels)
        except Exception:
            metrics["log_loss"] = np.nan

    return metrics, y_pred


def print_metrics(title, metrics):
    print(f"\n{title}")
    for k, v in metrics.items():
        print(f"{k:>20}: {v:.5f}")


def build_best_pipeline(vectorizer_kind, best_params):
    '''Формирует пайплайн с лучшими гиперпараметрами (или гиперпараметрами по умолчанию).'''
    if best_params is not None:
        ngram_range = (best_params["ngram_low"], best_params["ngram_high"])

    if vectorizer_kind == "count":
        vectorizer = CountVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            lowercase=False,
            token_pattern=None,
            ngram_range=ngram_range if best_params is not None else (1,1),
            min_df=best_params["min_df"] if best_params is not None else 1,
            max_df=best_params["max_df"] if best_params is not None else 1.0,
            max_features=best_params["max_features"] if best_params is not None else None,
            stop_words=None,
        )
    else:
        vectorizer = TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            lowercase=False,
            token_pattern=None,
            ngram_range=ngram_range if best_params is not None else (1,1),
            min_df=best_params["min_df"] if best_params is not None else 1,
            max_df=best_params["max_df"] if best_params is not None else 1.0,
            max_features=best_params["max_features"] if best_params is not None else None,
            sublinear_tf=best_params.get("sublinear_tf", False) if best_params is not None else False,
            stop_words=None,
        )

    clf = LogisticRegression(
        solver="saga",
        penalty="l2",
        C=best_params["C"] if best_params is not None else 1.0,
        class_weight=best_params["class_weight"] if best_params is not None else None,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        multi_class="multinomial",
    )

    return Pipeline([
        ("vec", vectorizer),
        ("clf", clf),
    ])


def make_pipeline(trial, vectorizer_kind):
    '''Используется для построения пайплайна с перебором гиперпаратров в optuna.'''
    ngram_low = trial.suggest_categorical("ngram_low", [1])
    ngram_high = trial.suggest_categorical("ngram_high", [1, 2])

    min_df = trial.suggest_categorical("min_df", [2, 3, 5])
    max_df = trial.suggest_float("max_df", 0.85, 0.98)
    max_features = trial.suggest_categorical("max_features", [30000, 50000, 80000])
    use_stop_words = trial.suggest_categorical("use_stop_words", [False, True])

    if vectorizer_kind == "count":
        vectorizer = CountVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            lowercase=False,
            token_pattern=None,
            ngram_range=(ngram_low, ngram_high),
            min_df=min_df,
            max_df=max_df,
            max_features=max_features,
            stop_words=None,
        )
    elif vectorizer_kind == "tfidf":
        vectorizer = TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            lowercase=False,
            token_pattern=None,
            ngram_range=(ngram_low, ngram_high),
            min_df=min_df,
            max_df=max_df,
            max_features=max_features,
            sublinear_tf=trial.suggest_categorical("sublinear_tf", [False, True]),
            stop_words=None,
        )
    else:
        raise ValueError("vectorizer_kind must be 'count' or 'tfidf'")

    clf = LogisticRegression(
        solver="saga",
        penalty="l2",
        C=trial.suggest_float("C", 1e-2, 10.0, log=True),
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        max_iter=1000,
        tol=1e-3,
        n_jobs=None,          # у saga обычно не ускоряет, поэтому не трогаем
        random_state=42,
        multi_class="multinomial",
    )

    return Pipeline([
        ("vec", vectorizer),
        ("clf", clf),
    ])


def objective_factory(vectorizer_kind):
    def objective(trial):
        pipe = make_pipeline(trial, vectorizer_kind)
        scores = cross_val_score(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring="f1_macro",
            n_jobs=-1,
        )
        return scores.mean()
    return objective

def run_search(vectorizer_kind, n_trials=10):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_factory(vectorizer_kind), n_trials=n_trials)
    return study

# Простые векторизации

## Обучение

In [26]:
pipeline_count = build_best_pipeline("count", None)
pipeline_tfidf = build_best_pipeline("tf-idf", None)

pipeline_count.fit(X_train, y_train)
pipeline_tfidf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('vec',
                 TfidfVectorizer(lowercase=False, token_pattern=None,
                                 tokenizer=<method 'split' of 'str' objects>)),
                ('clf',
                 LogisticRegression(max_iter=1000, multi_class='multinomial',
                                    random_state=42, solver='saga',
                                    tol=0.001))])

## Валидация

In [27]:
# проверим качество на eval-датасете
count_eval_metrics, count_eval_pred = evaluate_model(
    pipeline_count, X_eval, y_eval, labels=np.arange(df["target"].nunique())
)
tfidf_eval_metrics, tfidf_eval_pred = evaluate_model(
    pipeline_tfidf, X_eval, y_eval, labels=np.arange(df["target"].nunique())
)

print_metrics("CountVectorizer — eval metrics", count_eval_metrics)
print_metrics("TF-IDF — eval metrics", tfidf_eval_metrics)

print("\nClassification report for CountVectorizer on eval:")
print(classification_report(y_eval, count_eval_pred, digits=4, zero_division=0))

print("\nClassification report for TF-IDF on eval:")
print(classification_report(y_eval, tfidf_eval_pred, digits=4, zero_division=0))


CountVectorizer — eval metrics
            accuracy: 0.80825
   balanced_accuracy: 0.74645
     precision_macro: 0.78257
        recall_macro: 0.74645
            f1_macro: 0.76191
         f1_weighted: 0.80639
            log_loss: 0.63468

TF-IDF — eval metrics
            accuracy: 0.81098
   balanced_accuracy: 0.70570
     precision_macro: 0.80712
        recall_macro: 0.70570
            f1_macro: 0.73320
         f1_weighted: 0.80401
            log_loss: 0.60089

Classification report for CountVectorizer on eval:
              precision    recall  f1-score   support

           0     0.5612    0.3768    0.4509       207
           1     0.8152    0.8129    0.8140      1454
           2     0.8698    0.8166    0.8424       589
           3     0.6381    0.6011    0.6190       757
           4     0.7489    0.7038    0.7256      1212
           5     0.8689    0.8574    0.8631      1438
           6     0.7924    0.8305    0.8110      3682
           7     0.8169    0.8181    0.8

## Подбор гиперпараметров

In [135]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# count_study = run_search("count", n_trials=10)
tfidf_study = run_search("tfidf", n_trials=10)

print("\nBest CV score (TfidfVectorizer):", tfidf_study.best_value)
print("Best params (TfidfVectorizer):", tfidf_study.best_params)

# tfidf_model = build_best_pipeline("tfidf", tfidf_study.best_params)


Best CV score (TfidfVectorizer): 0.7603068606730788
Best params (TfidfVectorizer): {'ngram_low': 1, 'ngram_high': 2, 'min_df': 2, 'max_df': 0.8849152077396094, 'max_features': 80000, 'use_stop_words': True, 'sublinear_tf': False, 'C': 2.580542314754006, 'class_weight': 'balanced'}


По-хорошему нужно провести больше итераций поиска, полученные лучшие гиперпараметры использовать в дальнейшем для обучения и тестирования модели.

# Word2vec gensim

## Обучение эмбеддингов

In [36]:
import os
import multiprocessing
os.cpu_count(), multiprocessing.cpu_count()

(2, 2)

In [40]:
# будем пытаться обучать word2vec на всем доступном массиве данных
sentences = df["processed_text"].fillna("").apply(str.split).tolist()
sentences = [sent for sent in sentences if len(sent) > 0]

In [45]:
from gensim.models.callbacks import CallbackAny2Vec
import time

class EpochLogger(CallbackAny2Vec):
    def __init__(self, total_epochs):
        self.epoch = 0
        self.total_epochs = total_epochs
        self.start_time = time.time()

    def on_epoch_begin(self, model):
        print(f"Epoch {self.epoch+1}/{self.total_epochs} started")

    def on_epoch_end(self, model):
        elapsed = time.time() - self.start_time
        print(f"Epoch {self.epoch+1}/{self.total_epochs} finished | elapsed: {elapsed:.1f}s")
        self.epoch += 1

In [47]:
N_EPOCHS = 10

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100, # оставим пока так, так как датасет маленький
    window=5,
    min_count=5, # оставим по умолчанию
    sg=1, # skip-gram
    negative=10, # опять же из-за маленькости датасета возьмем значение чуть больше, чем по умолчанию
    sample=1e-3, # сэмплирование частых слов также оставим по умолчанию
    epochs=N_EPOCHS, # возьмем для начала небольшое число эпох
    workers=2,
    seed=42,
    callbacks=[EpochLogger(N_EPOCHS)]
)

Epoch 1/10 started
Epoch 1/10 finished | elapsed: 179.2s
Epoch 2/10 started
Epoch 2/10 finished | elapsed: 351.1s
Epoch 3/10 started
Epoch 3/10 finished | elapsed: 524.7s
Epoch 4/10 started
Epoch 4/10 finished | elapsed: 697.4s
Epoch 5/10 started
Epoch 5/10 finished | elapsed: 865.7s
Epoch 6/10 started
Epoch 6/10 finished | elapsed: 1035.6s
Epoch 7/10 started
Epoch 7/10 finished | elapsed: 1204.6s
Epoch 8/10 started
Epoch 8/10 finished | elapsed: 1375.5s
Epoch 9/10 started
Epoch 9/10 finished | elapsed: 1546.7s
Epoch 10/10 started
Epoch 10/10 finished | elapsed: 1716.2s


## Визуальная оценка качества эмбеддингов

Сходство с представленным словом:

In [49]:
w2v_model.wv.most_similar(positive=['апрель'], topn=5)

[('май', 0.9630777835845947),
 ('февраль', 0.962674617767334),
 ('июль', 0.9578331112861633),
 ('март', 0.9561031460762024),
 ('октябрь', 0.9558127522468567)]

In [60]:
w2v_model.wv.most_similar(positive=['автобус'], topn=5)

[('рейсовый', 0.8323673009872437),
 ('микроавтобус', 0.8212802410125732),
 ('кавза', 0.7932042479515076),
 ('икарус', 0.7857819199562073),
 ('паз', 0.7805033326148987)]

Лишнее слово в списке:

In [65]:
w2v_model.wv.doesnt_match(['футбол','теннис','баскетбол','волейбол','арбуз'])

'арбуз'

In [68]:
w2v_model.wv.doesnt_match(['кот','собака','метро','лошадь','слон'])

'метро'

## Обучение и валидация модели на обученных эмбеддингах

In [100]:
if hasattr(pipeline_tfidf[0], 'vocabulary_'):
    print("Vectorizer is fitted")

Vectorizer is fitted


In [103]:
pipeline_tfidf[0].get_feature_names_out()

array(['аа', 'ааа', 'аавик', ..., 'ёмкость', 'ёрзать', 'ёрничать'],
      dtype=object)

In [84]:
def document_vector(text, model, vector_size):
    tokens = text.split()
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(vector_size, dtype=np.float32)

    # будем просто усреднять вектора слов текста
    # (можно брать tf-idf-взвешенное среднее)
    return np.mean(vectors, axis=0)


vector_size = w2v_model.vector_size

X_w2v_train = np.vstack([
    document_vector(text, w2v_model, vector_size)
    for text in X_train
])

X_w2v_eval = np.vstack([
    document_vector(text, w2v_model, vector_size)
    for text in X_eval
])

In [94]:
clf_w2v = LogisticRegression(
    solver="saga",
    penalty="l2",
    max_iter=1000,
    tol=1e-3,
    random_state=42,
    multi_class="multinomial",
)
clf_w2v.fit(X_w2v_train, y_train)

w2v_eval_metrics, w2v_eval_pred = evaluate_model(
    clf_w2v, X_w2v_eval, y_eval, labels=np.arange(df["target"].nunique())
)
w2v_eval_metrics

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


{'accuracy': 0.7871457489878543,
 'balanced_accuracy': np.float64(0.6867675825955745),
 'precision_macro': 0.7434651088035461,
 'recall_macro': 0.6867675825955745,
 'f1_macro': 0.7002620770816947,
 'f1_weighted': 0.7792173111339998,
 'log_loss': 0.6282759019621829}

In [92]:
tfidf_eval_metrics

{'accuracy': 0.8109817813765182,
 'balanced_accuracy': np.float64(0.7056988768977015),
 'precision_macro': 0.8071186156700828,
 'recall_macro': 0.7056988768977015,
 'f1_macro': 0.7332038600580281,
 'f1_weighted': 0.804011686253485,
 'log_loss': 0.6008942957390814}

Метрики на валидации получились чуть хуже в сравнении с tf-idf. По-хорошему, нужно взять побольше датасет для кросс-валидации и аккуратно подобрать гиперпараметры для word2vec-модели, что займет достаточно много времени.

Можно попробовать использовать не простое усреднение векторов текста, а усреднение с учетом tf-idf-весов.

In [104]:
def document_vector_tfidf(text, w2v_model, tfidf_vectorizer, tfidf_vocab):
    tokens = text.split()
    vector_size = w2v_model.vector_size

    tfidf_vec = tfidf_vectorizer.transform([text])
    weights = tfidf_vec.toarray()[0]

    doc_vector = np.zeros(vector_size, dtype=np.float32)
    weight_sum = 0.0

    for token in tokens:
        if token in w2v_model.wv and token in tfidf_vocab:
            idx = tfidf_vocab[token]
            w = weights[idx]

            if w > 0:
                doc_vector += w * w2v_model.wv[token]
                weight_sum += w

    if weight_sum > 0:
        doc_vector /= weight_sum

    return doc_vector

In [107]:
feature_names = pipeline_tfidf[0].get_feature_names_out()
tfidf_vocab = {word: i for i, word in enumerate(feature_names)}

X_w2v_train_new = np.vstack([
    document_vector_tfidf(text, w2v_model, pipeline_tfidf[0], tfidf_vocab)
    for text in X_train
])
X_w2v_eval_new = np.vstack([
    document_vector_tfidf(text, w2v_model, pipeline_tfidf[0], tfidf_vocab)
    for text in X_eval
])

In [110]:
clf_w2v_new = LogisticRegression(
    solver="saga",
    penalty="l2",
    max_iter=1000,
    tol=1e-3,
    random_state=42,
    multi_class="multinomial",
)
clf_w2v_new.fit(X_w2v_train_new, y_train)

w2v_eval_metrics_new, w2v_eval_pred_new = evaluate_model(
    clf_w2v_new, X_w2v_eval_new, y_eval, labels=np.arange(df["target"].nunique())
)
w2v_eval_metrics_new

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


{'accuracy': 0.7717105263157895,
 'balanced_accuracy': np.float64(0.6717306959387965),
 'precision_macro': 0.7298167749494099,
 'recall_macro': 0.6717306959387965,
 'f1_macro': 0.6844124662822693,
 'f1_weighted': 0.76368460096968,
 'log_loss': 0.6970975174179544}

In [111]:
w2v_eval_metrics

{'accuracy': 0.7871457489878543,
 'balanced_accuracy': np.float64(0.6867675825955745),
 'precision_macro': 0.7434651088035461,
 'recall_macro': 0.6867675825955745,
 'f1_macro': 0.7002620770816947,
 'f1_weighted': 0.7792173111339998,
 'log_loss': 0.6282759019621829}

Как видим, в данном случае этот метод прироста не дал.

# Navec

In [115]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

--2026-04-21 19:00:37--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  14.6MB/s    in 1.7s    

2026-04-21 19:00:39 (14.6 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [120]:
navec = Navec.load("navec_news_v1_1B_250K_300d_100q.tar")
print("Embedding dim:", navec['москва'].shape[0])

def document_vector_navec(text, navec_model):
    tokens = str(text).split()
    vectors = []

    for token in tokens:
        if token in navec_model:
            vectors.append(navec_model[token])

    if len(vectors) == 0:
        return np.zeros(navec_model["<pad>"].shape[0], dtype=np.float32)

    return np.mean(vectors, axis=0)

X_navec_train = np.vstack([
    document_vector_navec(text, navec)
    for text in X_train
])

X_navec_eval = np.vstack([
    document_vector_navec(text, navec)
    for text in X_eval
])

clf_navec = LogisticRegression(
    solver="saga",
    penalty="l2",
    max_iter=1000,
    tol=1e-3,
    random_state=42,
    multi_class="multinomial",
)
clf_navec.fit(X_navec_train, y_train)

navec_eval_metrics, navec_eval_pred = evaluate_model(
    clf_navec, X_navec_eval, y_eval, labels=np.arange(df["target"].nunique())
)
navec_eval_metrics


Embedding dim: 300


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


{'accuracy': 0.7865384615384615,
 'balanced_accuracy': np.float64(0.6870683165259378),
 'precision_macro': 0.7410404006986049,
 'recall_macro': 0.6870683165259378,
 'f1_macro': 0.7005070007644313,
 'f1_weighted': 0.7792289516222904,
 'log_loss': 0.638025052839598}

In [122]:
w2v_eval_metrics

{'accuracy': 0.7871457489878543,
 'balanced_accuracy': np.float64(0.6867675825955745),
 'precision_macro': 0.7434651088035461,
 'recall_macro': 0.6867675825955745,
 'f1_macro': 0.7002620770816947,
 'f1_weighted': 0.7792173111339998,
 'log_loss': 0.6282759019621829}

Качество векторов в данном случае сопоставимо.

# Финальная проверка качества на отложенной выборке

In [130]:
# CountVectorizer
count_test_metrics, count_test_pred = evaluate_model(
    pipeline_count, X_test, y_test, labels=np.arange(df["target"].nunique())
)
count_test_metrics

{'accuracy': 0.8058296644906634,
 'balanced_accuracy': np.float64(0.7455478835120011),
 'precision_macro': 0.7850758151677254,
 'recall_macro': 0.7455478835120011,
 'f1_macro': 0.7631214398113649,
 'f1_weighted': 0.804355325767365,
 'log_loss': 0.6488583216857531}

In [132]:
# TF_IDF
tfidf_test_metrics, tfidf_test_pred = evaluate_model(
    pipeline_tfidf, X_test, y_test, labels=np.arange(df["target"].nunique())
)
tfidf_test_metrics

{'accuracy': 0.809017762258995,
 'balanced_accuracy': np.float64(0.7060342943362478),
 'precision_macro': 0.8103677669419463,
 'recall_macro': 0.7060342943362478,
 'f1_macro': 0.735520112096717,
 'f1_weighted': 0.8027462426908074,
 'log_loss': 0.6128528423436365}

In [95]:
# word2vec-gensim обученные эмбеддинги
X_w2v_test = np.vstack([
    document_vector(text, w2v_model, vector_size)
    for text in X_test
])

w2v_test_metrics, w2v_test_pred = evaluate_model(
    clf_w2v, X_w2v_test, y_test, labels=np.arange(df["target"].nunique())
)
w2v_test_metrics

{'accuracy': 0.7780476696523455,
 'balanced_accuracy': np.float64(0.6720321033091347),
 'precision_macro': 0.7216399178128863,
 'recall_macro': 0.6720321033091347,
 'f1_macro': 0.6859810233490288,
 'f1_weighted': 0.7703265862149906,
 'log_loss': 0.6452239511860592}

In [129]:
# navec подгруженные вектора
X_navec_test = np.vstack([
    document_vector_navec(text, navec)
    for text in X_test
])

navec_test_metrics, navec_test_pred = evaluate_model(
    clf_navec, X_navec_test, y_test, labels=np.arange(df["target"].nunique())
)
navec_test_metrics

{'accuracy': 0.7763777136784575,
 'balanced_accuracy': np.float64(0.6750280779753084),
 'precision_macro': 0.7181363697801149,
 'recall_macro': 0.6750280779753084,
 'f1_macro': 0.6876103855388197,
 'f1_weighted': 0.7692764845918999,
 'log_loss': 0.6525719611520998}

Как мы видим, в данной простой задачке наилучшее качество на отложенной выборке без перебора гиперпараметров демонстрируют более простые модели, такие как TF-IDF и CountVectorizer.